# 在 Eino 中创建 Tool，主要有四种方式。最推荐的是 utils.InferTool，它能直接从 Go 结构体的 tag 推导出参数 Schema，避免手动维护两份定义。

    方式一：使用 InferTool（推荐）
        这是最简洁的方式。你只需定义一个参数结构体，用 jsonschema tag 描述字段约束，InferTool 会自动生成 ToolInfo。

        ```go
        import (
            "context"
            "github.com/cloudwego/eino/components/tool"
            "github.com/cloudwego/eino/components/tool/utils"
        )

        type WeatherParams struct {
            City string `json:"city" jsonschema:"required,description=城市名称"`
            Unit string `json:"unit" jsonschema:"enum=celsius,enum=fahrenheit,description=温度单位"`
        }

        func getWeather(ctx context.Context, params *WeatherParams) (string, error) {
            return "晴，25°C", nil
        }

        // 创建工具
        weatherTool, err := utils.InferTool(
            "get_weather",
            "查询指定城市的天气",
            getWeather,
        )
        ```

    关键点：json tag 控制参数名，jsonschema tag 中的 required 标记必填，description 提供字段说明，enum 限定可选值。

    方式二：使用 NewTool（手动定义 Schema）
        当参数结构无法用 Go struct 表达（例如需要动态构建 Schema），或你希望完全手动控制时使用。

        ```go
        import "github.com/cloudwego/eino/schema"

        toolInfo := &schema.ToolInfo{
            Name: "get_weather",
            Desc: "查询指定城市的天气",
            ParamsOneOf: schema.NewParamsOneOfByParams(map[string]*schema.ParameterInfo{
                "city": {Type: "string", Desc: "城市名称", Required: true},
            }),
        }

        weatherTool := utils.NewTool(toolInfo, getWeather)
        ```
        缺点是 ToolInfo 中的参数定义与 getWeather 的参数结构是分开维护的，容易不一致。

        方式三：实现 Tool 接口
            适用于需要完全自定义逻辑的场景（如动态 Info、特殊序列化处理）。以 InvokableTool 为例：

            ```go
            type MyTool struct{}

            func (t *MyTool) Info(ctx context.Context) (*schema.ToolInfo, error) {
                return &schema.ToolInfo{ /* ... */ }, nil
            }

            func (t *MyTool) InvokableRun(ctx context.Context, argumentsInJSON string, opts ...tool.Option) (string, error) {
                // 需要自行反序列化 argumentsInJSON
                return "result", nil
            }
            ```
            这种方式最灵活，但需要手动处理 JSON 反序列化。

        方式四：使用官方封装工具
            Eino 生态（eino-ext）提供了开箱即用的工具，直接引入即可使用。例如 DuckDuckGo 搜索：

            ```go
            import "github.com/bytedance/eino-ext/components/tool/duckduckgo"

            searchTool, err := duckduckgo.NewTool(ctx, &duckduckgo.Config{ /* ... */ })
            ```

## 补充：多模态与流式工具
        如果你需要返回图片等多模态结果，可使用 utils.InferEnhancedTool；需要流式返回多模态内容，使用 utils.InferEnhancedStreamTool。它们与 InferTool 用法类似，只是函数签名不同。
        选型建议：优先用 InferTool；需要动态 Schema 或特殊控制时用 NewTool；有极端定制需求时实现接口；通用能力优先找官方封装。